In [1]:
# Run this cell first. It installs the packages this notebook uses.
%pip install -q pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


# 02. Midterm elections since 1978

## The ideas first, in plain words

**Midterm election.** The congressional election held halfway through a
president's four year term, in the even years when there is no presidential
election. All 435 House seats are on the ballot, plus about a third of the
100 Senate seats, plus many governor and state offices.

**The president's party.** The party of the sitting president. In a midterm,
voters are often judging that president, even though the president is not
on the ballot.

**Seat change.** How many seats the president's party gained or lost
compared with the Congress before the midterm. A negative number means the
president's party lost seats.

**Unified government and divided government.** Unified government means one
party controls the presidency, the House, and the Senate at the same time.
That combination is often called a trifecta. Divided government means at
least one chamber is controlled by the other party.

**Seat counts used here.** House and Senate seat counts come from the
official party division tables (Clerk of the House via Wikipedia, and
senate.gov). Those tables record the party balance at the start of each
Congress, so a seat that changed hands in a special election or a party
switch during a Congress can make a start of Congress number differ by a
seat or two from the election night number. The pattern below is the same
either way.

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt

PROC = os.path.join('..', 'data', 'processed')
CHART_DIR = os.path.join('..', 'charts')
os.makedirs(CHART_DIR, exist_ok=True)
plt.rcParams['figure.dpi'] = 120
DEM = '#1f77b4'
REP = '#d62728'
GREY = '#7f7f7f'

def savefig(name):
    path = os.path.join(CHART_DIR, name)
    plt.savefig(path, bbox_inches='tight')
    plt.close()
    print('saved', path)

In [3]:
mid = pd.read_csv(os.path.join(PROC, 'midterm_history.csv'))
hdiv = pd.read_csv(os.path.join(PROC, 'house_division.csv'))
sdiv = pd.read_csv(os.path.join(PROC, 'senate_division.csv'))
print(mid[['midterm_year', 'president', 'president_party', 'house_before', 'house_after',
           'house_change', 'senate_before', 'senate_after', 'senate_change',
           'unified_before', 'unified_after']].to_string(index=False))

 midterm_year         president president_party  house_before  house_after  house_change  senate_before  senate_after  senate_change  unified_before  unified_after
         1978      Jimmy Carter               D           292          277           -15             61            58             -3            True           True
         1982     Ronald Reagan               R           192          165           -27             53            54              1           False          False
         1986     Ronald Reagan               R           181          177            -4             53            45             -8           False          False
         1990 George H. W. Bush               R           175          167            -8             45            44             -1           False          False
         1994      Bill Clinton               D           258          204           -54             57            47            -10            True          False
         1998   

## The House: the president's party usually loses seats

This is the most reliable pattern in American midterm elections. In 10 of
the 12 midterms from 1978 to 2022, the president's party lost House seats.
The average loss was 21.2 seats, and the median loss was 14 seats. The two
exceptions are 1998, when Bill Clinton's Democrats gained 5 seats, and
2002, when George W. Bush's Republicans gained 8 seats.

In [4]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = [REP if v >= 0 else DEM for v in mid['house_change']]
ax.bar(mid['midterm_year'].astype(str), mid['house_change'], color=colors)
ax.axhline(0, color='black', linewidth=1)
ax.set_ylabel('House seat change for the president party')
ax.set_xlabel('Midterm year')
ax.set_title('House seat change for the president party, 1978 to 2022')
ax.grid(axis='y', alpha=0.3)
savefig('c02_house_change.png')
print('Midterms where the president party lost House seats:', (mid['house_change'] < 0).sum(), 'of', len(mid))
print('Average House change: %.1f seats.' % mid['house_change'].mean())
print('Median House change: %.1f seats.' % mid['house_change'].median())
print('Largest House loss: 2010, Democrats lost 64 seats.')
print('Largest House gain: 2002, Republicans gained 8 seats.')

saved ../charts/c02_house_change.png
Midterms where the president party lost House seats: 10 of 12
Average House change: -21.2 seats.
Median House change: -14.0 seats.
Largest House loss: 2010, Democrats lost 64 seats.
Largest House gain: 2002, Republicans gained 8 seats.


## The Senate: smaller moves, and three gains

Senate changes are smaller because only about a third of seats are up each
time, and the map of which seats are up matters a lot. The president's
party gained Senate seats in 3 of 12 midterms (1982, 2002, 2018), broke even
in 2 (1998, 2022), and lost seats in 7. The average Senate change was a
loss of 3.25 seats. The biggest Senate loss was 1994, when Clinton's
Democrats lost 10 seats.

In [5]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = [REP if v >= 0 else DEM for v in mid['senate_change']]
ax.bar(mid['midterm_year'].astype(str), mid['senate_change'], color=colors)
ax.axhline(0, color='black', linewidth=1)
ax.set_ylabel('Senate seat change for the president party')
ax.set_xlabel('Midterm year')
ax.set_title('Senate seat change for the president party, 1978 to 2022')
ax.grid(axis='y', alpha=0.3)
savefig('c02_senate_change.png')
print('Senate gains:', (mid['senate_change'] > 0).sum(), '| even:', (mid['senate_change'] == 0).sum(),
      '| losses:', (mid['senate_change'] < 0).sum())
print('Average Senate change: %.2f seats.' % mid['senate_change'].mean())

saved ../charts/c02_senate_change.png
Senate gains: 3 | even: 2 | losses: 7
Average Senate change: -3.25 seats.


## House seats by party over time

Each line is one party's total House seats at the start of each Congress.
The lines cross when control flips. The big flips are 1994 (Democrats to
Republicans), 2006 (Republicans to Democrats), 2010 (Democrats to
Republicans), and 2018 (Republicans to Democrats). Every one of those flips
happened in a midterm.

In [6]:
hdiv['start_year'] = hdiv['years'].str.slice(0, 4).astype(int)
plt.figure(figsize=(10, 5))
plt.plot(hdiv['start_year'], hdiv['dem'], marker='o', color=DEM, label='Democrats')
plt.plot(hdiv['start_year'], hdiv['rep'], marker='o', color=REP, label='Republicans')
plt.axhline(218, color=GREY, linestyle='--', linewidth=1, label='Majority = 218')
plt.ylabel('House seats')
plt.xlabel('Congress start year')
plt.title('House seats by party, 1977 to 2027')
plt.legend()
plt.grid(alpha=0.3)
savefig('c02_house_seats_time.png')

saved ../charts/c02_house_seats_time.png


## Senate seats by party over time

The Senate moves more slowly. The lines cross in 1980 (Democrats to
Republicans), 1986 (Republicans to Democrats), 1994 (Democrats to
Republicans), 2006 (Republicans to Democrats), and 2014 (Democrats to
Republicans). The 2001 to 2003 period is the famous 50 to 50 Senate, which
Democrats controlled after Senator Jim Jeffords left the Republican Party
and after the vice president's tie breaking vote is counted.

In [7]:
sdiv['start_year'] = sdiv['years'].str.slice(0, 4).astype(int)
plt.figure(figsize=(10, 5))
plt.plot(sdiv['start_year'], sdiv['dem'], marker='o', color=DEM, label='Democrats')
plt.plot(sdiv['start_year'], sdiv['rep'], marker='o', color=REP, label='Republicans')
plt.axhline(51, color=GREY, linestyle='--', linewidth=1, label='Majority = 51')
plt.ylabel('Senate seats')
plt.xlabel('Congress start year')
plt.title('Senate seats by party, 1977 to 2027')
plt.legend()
plt.grid(alpha=0.3)
savefig('c02_senate_seats_time.png')

saved ../charts/c02_senate_seats_time.png


## The trifecta pattern

A president who enters a midterm holding the presidency, the House, and
the Senate almost always leaves without at least one of them. From 1978 to
2022, six presidents entered a midterm with a trifecta: Carter in 1978,
Clinton in 1994, George W. Bush in 2006, Obama in 2010, Trump in 2018, and
Biden in 2022. Five of the six lost the trifecta in that midterm. The only
one who kept it was Carter in 1978. And 2002 is the mirror image: George W.
Bush entered that midterm without a trifecta, because Democrats controlled
the Senate, and left with one, because Republicans gained the Senate.

## Turnout in midterms

Midterms draw fewer voters than presidential elections. Voting eligible
population turnout was 36.7 percent in 2014, 50.1 percent in 2018, and
46.1 percent in 2022. Compare that with 65.28 percent in the 2020
presidential election. Lower turnout is one reason midterm electorates look
different from presidential electorates, and it is part of why the
president's party usually struggles.

## What to take into the next chapter

The midterm penalty is real but it is not a law of nature. It failed in
1998, 2002, and in the Senate in 2018. The next chapter asks what actually
changes for a president after a midterm, and how far the numbers can show
it.